In [1]:
# import packages
!pip install anndata scanpy session_info GEOparse seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 68.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2,

In [2]:
# Load packages
from __future__ import annotations
import anndata as ad
import tarfile as tf
import scanpy as sc
import session_info
import GEOparse
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
# -- Fetch GSE data from GEO
gse = GEOparse.get_GEO(
    geo="GSE179640",
    destdir="geo_download",
    include_data=True
)

gse.download_supplementary_files()

02-Jun-2026 05:08:22 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE179nnn/GSE179640/soft/GSE179640_family.soft.gz to geo_download/GSE179640_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE179nnn/GSE179640/soft/GSE179640_family.soft.gz to geo_download/GSE179640_family.soft.gz
100%|██████████| 8.49k/8.49k [00:00<00:00, 19.3kB/s]
02-Jun-2026 05:08:23 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
02-Jun-2026 05:08:23 DEBUG downloader - Moving /tmp/tmp1ii8kfb8 to /content/geo_download/GSE179640_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp1ii8kfb8 to /content/geo_download/GSE179640_family.soft.gz
02-Jun-2026 05:08:23 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE179nnn/GSE179640/soft/GSE179640_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE179nnn/GSE179640/soft/GSE179640_family.soft.gz
02-Jun-2026 05:08:23 INFO GEOpa

{'GSM6102532': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102532/suppl/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5': '/content/GSE179640_Supp/Supp_GSM6102532_Control_Patient_1_-_Control_Endometrium/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102533': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102533/suppl/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5': '/content/GSE179640_Supp/Supp_GSM6102533_Control_Patient_2_-_Control_Endometrium/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102534': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102534/suppl/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5': '/content/GSE179640_Supp/Supp_GSM6102534_Control_Patient_3_-_Control_Endometrium/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102537': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102537/suppl/GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5': '/content/GSE179640_Supp/Supp_GSM6102537_Endometriosis_

In [5]:
os.listdir("geo_download")

['GSE179640_family.soft.gz']

In [6]:
for root, dirs, files in os.walk("/content/GSE179640_Supp"):
    print(root)
    for f in files:
        print("   ", f)

/content/GSE179640_Supp
/content/GSE179640_Supp/Supp_GSM6595257_Endometriosis_Patient_5_-_Ectopic_Adjacent
    GSM6595257_E05_EcPA_filtered_feature_bc_matrix.h5
/content/GSE179640_Supp/Supp_GSM6102585_Endometriosis_Patient_16_-_Ectopic_Ovary_-_bulk
    GSM6102585_E16_EcO_bulk.featurecounts.txt.gz
/content/GSE179640_Supp/Supp_GSM6595255_Endometriosis_Patient_4_-_Ectopic_Adjacent
    GSM6595255_E04_EcPA_filtered_feature_bc_matrix.h5
/content/GSE179640_Supp/Supp_GSM6595260_Endometriosis_Patient_9_-_Ectopic_Adjacent-2
    GSM6595260_E09_EcPA2_filtered_feature_bc_matrix.h5
/content/GSE179640_Supp/Supp_GSM6595251_Endometriosis_Patient_2_-_Ectopic_Adjacent
    GSM6595251_E02_EcPA_filtered_feature_bc_matrix.h5
/content/GSE179640_Supp/Supp_GSM6595249_Endometriosis_Patient_1_-_Ectopic_Adjacent
    GSM6595249_E01_EcPA_filtered_feature_bc_matrix.h5
/content/GSE179640_Supp/Supp_GSM6102587_Endometriosis_Patient_16_-_Eutopic_-_bulk
    GSM6102587_E16_EuE_bulk.featurecounts.txt.gz
/content/GSE179640_S

In [7]:
# -- Set up metadata

# samples to exclude
EXCLUDE = {"EOR", "EcPA"}

# metadata map keyed by sample_id extracted from filename
metadata = {
    # controls
    "GSM6102532_C01_Ctrl": {"patient_id": "C01", "tissue_type": "healthy",          "condition": "control",         "lesion_site": None},
    "GSM6102533_C02_Ctrl": {"patient_id": "C02", "tissue_type": "healthy",          "condition": "control",         "lesion_site": None},
    "GSM6102534_C03_Ctrl": {"patient_id": "C03", "tissue_type": "healthy",          "condition": "control",         "lesion_site": None},

    # eutopic
    "GSM6102537_E01_EuE":  {"patient_id": "E01", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102540_E02_EuE":  {"patient_id": "E02", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102543_E03_EuE":  {"patient_id": "E03", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102546_E04_EuE":  {"patient_id": "E04", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102549_E05_EuE":  {"patient_id": "E05", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102551_E06_EuE":  {"patient_id": "E06", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102554_E07_EuE":  {"patient_id": "E07", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102555_E08_EuE":  {"patient_id": "E08", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102560_E09_EuE":  {"patient_id": "E09", "tissue_type": "eutopic",          "condition": "endometriosis",   "lesion_site": None},

    # ectopic peritoneal
    "GSM6595248_E01_EcP":  {"patient_id": "E01", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595250_E02_EcP":  {"patient_id": "E02", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595252_E03_EcP":  {"patient_id": "E03", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595254_E04_EcP":  {"patient_id": "E04", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595256_E05_EcP":  {"patient_id": "E05", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102550_E06_EcP":  {"patient_id": "E06", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102553_E07_EcP":  {"patient_id": "E07", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595258_E09_EcP":  {"patient_id": "E09", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "peritoneal"},

    # ectopic ovarian
    "GSM6102552_E07_EcO":  {"patient_id": "E07", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102556_E09_EcO":  {"patient_id": "E09", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6595261_E10_EcO":  {"patient_id": "E10", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102562_E11_EcO":  {"patient_id": "E11", "tissue_type": "ectopic",          "condition": "endometriosis",   "lesion_site": "ovarian"},
}

In [8]:
# -- Collect all wanted files
h5_files = []

for root, dirs, files in os.walk("/content/GSE179640_Supp"):
    for f in files:
        if f.endswith(".h5"):
            h5_files.append(os.path.join(root, f))

In [9]:
# -- Set up object
adatas = []

for file in h5_files:
    sample_id = os.path.basename(file).replace("_filtered_feature_bc_matrix.h5", "")

# -- Exclude organoids and unwanted tissue types (Adjacent Peritoneium)

    if any(excl in sample_id for excl in EXCLUDE):
      print(f"Skipping: {sample_id}")
      continue

# -- Exclude samples without any metadata

    if sample_id not in metadata:
      print(
          f"Skipping {sample_id} - No metadata found."
          )

# -- Load data
    adata = sc.read_10x_h5(file)
    adata.var_names_make_unique()

# -- Add metadata to the object
    meta = metadata[sample_id]
    adata.obs["sample_id"] = sample_id
    adata.obs["patient_id"] = meta['patient_id']
    adata.obs["tissue_type"] = meta["tissue_type"]
    adata.obs["condition"] = meta["condition"]
    adata.obs["lesion_site"] = meta["lesion_site"]
    adata.obs["dataset"] = "GSE179640"

# -- Append sample name to cell barcodes
    adata.obs_names = [
        f"{sample_id}_{bc}"
        for bc in adata.obs_names
    ]

    print(f"{sample_id} loaded - {adata.n_obs} total cells")
    adatas.append(adata)

# -- Concat
combined = ad.concat(adatas, join = "outer")
combined.var_names_make_unique()

# -- Check
print(f"\nTotal cells: {combined.n_obs}")
print(combined.obs["tissue_type"].value_counts())
print(combined.obs["lesion_site"].value_counts())

Skipping: GSM6595257_E05_EcPA
Skipping: GSM6595255_E04_EcPA
Skipping: GSM6595260_E09_EcPA2
Skipping: GSM6595251_E02_EcPA
Skipping: GSM6595249_E01_EcPA


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102540_E02_EuE loaded - 5277 total cells
Skipping: GSM6102565_EOR03


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102560_E09_EuE loaded - 5255 total cells


...

GSM6102549_E05_EuE loaded - 3030 total cells


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102551_E06_EuE loaded - 5226 total cells


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


GSM6102550_E06_EcP loaded - 2494 total cells

Total cells: 119560
tissue_type
ectopic    60663
eutopic    41432
healthy    17465
Name: count, dtype: int64
lesion_site
peritoneal    31664
ovarian       28999
Name: count, dtype: int64


In [10]:
# -- Check to make sure every donor is accounted for
combined.obs.groupby(
    ["tissue_type",
     "patient_id"]).size().unstack(fill_value=0)

patient_id,C01,C02,C03,E01,E02,E03,E04,E05,E06,E07,E08,E09,E10,E11
tissue_type,,,,,,,,,,,,,,
ectopic,0,0,0,2961,5049,6691,3536,4043,2494,7060,0,8670,12788,7371
eutopic,0,0,0,7665,5277,5302,3819,3030,5226,3078,2780,5255,0,0
healthy,4133,7801,5531,0,0,0,0,0,0,0,0,0,0,0


In [13]:
# -- Save object for QC analysis
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -- Save to drive
combined.write_h5ad("/content/drive/MyDrive/endo-immune-atlas/data/processed/GSE179640_raw.h5ad")

Mounted at /content/drive


In [ ]:
## Session info
#session_info.show()